# Execução da camada Silver — ENEM por UF

Este notebook executa a etapa Silver para as edições de **1998 a 2025** para a UF definida em `UF_EXECUCAO`.

- Para processar outro estado, altere somente `UF_EXECUCAO` na célula 2.
- Cada UF é gravada em diretório próprio (`data/silver/uf=XX/`), sem misturar dados de estados.
- Parquets existentes e válidos daquela UF são reutilizados.
- Em 2024 e 2025, `participantes` e `resultados` permanecem separados.
- A execução para no primeiro erro.

Não altera Bronze nem Gold.

## 1. Configuração e imports

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Altere para a sigla desejada: PA, AP, AM, RR, etc.
UF_EXECUCAO = "RR"

RAIZ = Path.cwd()
if not (RAIZ / "src" / "enem_pipeline" / "config.py").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "src" / "enem_pipeline" / "config.py").exists(), (
    "Não foi possível localizar src/enem_pipeline/config.py. "
    "Abra o notebook na raiz do Projeto_ENEM ou dentro de notebooks/."
)

sys.path.insert(0, str(RAIZ / "src"))

from enem_pipeline.config import (
    PROJECT_ROOT, ANO_INICIAL, ANO_FINAL, SILVER_DIR,
    normalizar_uf, obter_diretorio_uf,
)
from enem_pipeline.silver import processar_ano_silver

UF_EXECUCAO = normalizar_uf(UF_EXECUCAO)
DIRETORIO_SILVER_UF = obter_diretorio_uf(SILVER_DIR, UF_EXECUCAO)

print("Raiz do projeto:", PROJECT_ROOT)
print("UF selecionada:", UF_EXECUCAO)
print("Período:", f"{ANO_INICIAL}–{ANO_FINAL}")
print("Destino Silver:", DIRETORIO_SILVER_UF)

Raiz do projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF selecionada: RR
Período: 1998–2025
Destino Silver: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR


## 2. Conferência dos parâmetros

In [2]:
assert ANO_INICIAL == 1998
assert ANO_FINAL == 2025
assert len(UF_EXECUCAO) == 2
assert (PROJECT_ROOT / "src" / "enem_pipeline" / "silver.py").exists()

ANOS = list(range(ANO_INICIAL, ANO_FINAL + 1))
print(f"UF: {UF_EXECUCAO}")
print(f"Anos a percorrer: {len(ANOS)}")
print(f"Fontes Silver esperadas: {len(ANOS) + 2} (2024 e 2025 têm duas bases cada)")

UF: RR
Anos a percorrer: 28
Fontes Silver esperadas: 30 (2024 e 2025 têm duas bases cada)


## 3. Executar Bronze/Staging → Silver

Execute esta célula uma única vez. A função recebe explicitamente `uf=UF_EXECUCAO`; os resultados ficam isolados no diretório dessa UF. Arquivos existentes e válidos são reutilizados, sem reprocessamento desnecessário.

In [3]:
resultados_silver = []

for posicao, ano in enumerate(ANOS, start=1):
    print(f"\n{'=' * 64}")
    print(f"[{posicao}/{len(ANOS)}] Processando edição {ano} — UF {UF_EXECUCAO}")
    print("=" * 64)

    try:
        resultados_ano = processar_ano_silver(ano, uf=UF_EXECUCAO)
        resultados_silver.extend(resultados_ano)

        for resultado in resultados_ano:
            origem = resultado.get("arquivo_origem", "fonte")
            status = resultado.get("status", "sem_status")
            destino = resultado.get("arquivo_saida", "")
            registros = resultado.get("total_registros")
            colunas = resultado.get("colunas_gravadas")

            print(f"Fonte: {origem}")
            print(f"Status: {status}")
            if destino:
                print(f"Saída: {destino}")
            if registros is not None:
                print(f"Registros: {registros:,} | Colunas: {colunas}")
            if resultado.get("validado") is False:
                raise RuntimeError(f"Validação não aprovada para {origem}, ano {ano}.")

    except Exception as erro:
        print(f"FALHA no ano {ano}: {type(erro).__name__}: {erro}")
        print("Execução interrompida; corrija a falha antes de continuar.")
        raise

print(f"\nExecução Silver concluída para todos os anos — UF {UF_EXECUCAO}.")


[1/28] Processando edição 1998 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=1998/MICRODADOS_ENEM_1998.csv (63,066,204 bytes)
Fonte: MICRODADOS_ENEM_1998.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1998/enem_1998.parquet
Registros: 967 | Colunas: 27

[2/28] Processando edição 1999 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=1999/MICRODADOS_ENEM_1999.csv (144,751,294 bytes)
Fonte: MICRODADOS_ENEM_1999.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=1999/enem_1999.parquet
Registros: 1,014 | Colunas: 28

[3/28] Processando edição 2000 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2000/MICRODADOS_ENEM_2000.csv (173,638,593 bytes)
Fonte: MICRODADOS_ENEM_2000.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fonte: MICRODADOS_ENEM_2001.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2001/enem_2001.parquet
Registros: 2,606 | Colunas: 35

[5/28] Processando edição 2002 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2002/MICRODADOS_ENEM_2002.csv (1,064,185,343 bytes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fonte: MICRODADOS_ENEM_2002.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2002/enem_2002.parquet
Registros: 5,497 | Colunas: 34

[6/28] Processando edição 2003 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2003/MICRODADOS_ENEM_2003.csv (996,760,902 bytes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (41 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fonte: PARTICIPANTES_2024.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2024/participantes.parquet
Registros: 12,695 | Colunas: 17
Fonte: RESULTADOS_2024.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2024/resultados.parquet
Registros: 12,695 | Colunas: 44

[28/28] Processando edição 2025 — UF RR
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2025/PARTICIPANTES_2025.csv (513,113,794 bytes)
Extraído e conferido: /home/akel/PycharmProjects/Projeto_ENEM/data/staging/ano=2025/RESULTADOS_2025.csv (2,115,499,721 bytes)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fonte: PARTICIPANTES_2025.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2025/participantes.parquet
Registros: 14,158 | Colunas: 17
Fonte: RESULTADOS_2025.csv
Status: processado_validado
Saída: /home/akel/PycharmProjects/Projeto_ENEM/data/silver/uf=RR/ano=2025/resultados.parquet
Registros: 14,158 | Colunas: 44

Execução Silver concluída para todos os anos — UF RR.


## 4. Resumo da execução

In [4]:
df_resultados_silver = pd.DataFrame(resultados_silver)

if df_resultados_silver.empty:
    print("A rotina não retornou resultados.")
else:
    colunas_resumo = [
        coluna for coluna in [
            "ano", "arquivo_origem", "arquivo_saida", "total_registros",
            "colunas_gravadas", "validado", "status", "csv_removido"
        ]
        if coluna in df_resultados_silver.columns
    ]
    display(df_resultados_silver[colunas_resumo])
    print("Fontes retornadas:", len(df_resultados_silver))
    if "validado" in df_resultados_silver.columns:
        print("Fontes validadas:", int(df_resultados_silver["validado"].fillna(False).sum()))
    if "status" in df_resultados_silver.columns:
        display(df_resultados_silver["status"].value_counts(dropna=False).rename_axis("status").to_frame("quantidade"))

,ano,arquivo_origem,arquivo_saida,total_registros,colunas_gravadas,validado,status,csv_removido
0,1998,MICRODADOS_ENEM_1998.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,967,27,True,processado_validado,True
1,1999,MICRODADOS_ENEM_1999.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,1014,28,True,processado_validado,True
2,2000,MICRODADOS_ENEM_2000.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,317,28,True,processado_validado,True
3,2001,MICRODADOS_ENEM_2001.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,2606,35,True,processado_validado,True
4,2002,MICRODADOS_ENEM_2002.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,5497,34,True,processado_validado,True
5,2003,MICRODADOS_ENEM_2003.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,6316,35,True,processado_validado,True
6,2004,MICRODADOS_ENEM_2004.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,4793,35,True,processado_validado,True
7,2005,MICRODADOS_ENEM_2005.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,5986,35,True,processado_validado,True
8,2006,MICRODADOS_ENEM_2006.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,2810,36,True,processado_validado,True
9,2007,MICRODADOS_ENEM_2007.csv,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,4207,36,True,processado_validado,True


Fontes retornadas: 30
Fontes validadas: 30


,quantidade
status,
processado_validado,30


## 5. Critério de conclusão

Considere a execução concluída quando a célula de processamento termina sem exceção, o resumo apresenta as **30 fontes esperadas** (26 de 1998–2023 e 4 de 2024–2025) e as fontes retornam `validado=True`.

Os dados são gravados em `data/silver/uf={UF_EXECUCAO}/`. Para processar outra UF, altere `UF_EXECUCAO` e execute novamente; não é necessário apagar nem reprocessar os dados de outras UFs.

Este notebook não executa a auditoria final da Gold nem modifica essa camada.